# QuickMapper: from any tabular file to RDF

This notebook is for the case where you have a measurement file but no
existing parser or schema for it.  Fill in a short mapping config and you
get a valid RDF graph back within minutes.

**Supported file formats:** CSV, TSV, TXT (auto-sniffed), Excel (.xlsx / .xls), Parquet

**What you provide:**
- Your data file
- A mapping config (dict or YAML) with two sections:
  - `metadata`: which fields to pull from the header block and what ontology property to assign
  - `columns`: which data columns to annotate with ontology class IRIs and QUDT units

**What you get back:**
- An RDF graph typed as `dcat:Dataset` (or any class you specify)
- One RDF property per extracted metadata field on the root node
- One ontology-annotated descriptor node per mapped column
- A pandas DataFrame with the full time-series data

The result is intentionally lightweight.  When the same file type recurs
often enough to warrant a formal schema, the `adding-a-parser.md` guide
explains how to graduate from a mapping config to a proper parser.

## What does a measurement file look like?

Most instrument files from materials testing and process monitoring equipment
follow a two-part layout.

**Header block (the "metadata")**

The first rows of the file record conditions that were set once before the test
started: the sample name, temperature, test speed, gauge length, testing standard,
and so on.  These values do not change during the test.  They describe *what was
being tested and how*.

**Data table**

Below the header block is a table where each row is one measurement point in time
and each column is one quantity the instrument recorded (force, displacement,
elongation, etc.).  This is the part that becomes a pandas DataFrame.

**How the mapping connects them**

The `metadata` section of your mapping selects which header-block rows to extract
and assigns each one an ontology property so it appears correctly in the knowledge
graph.  The `columns` section selects which data table columns to annotate with an
ontology class and a physical unit.  Columns you do not list are still returned in
the DataFrame with no ontology annotation.

In [1]:
%pip install -q semantic-transformers openpyxl pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv, json, pathlib, yaml
import rdflib
from semantic_transformers import QuickMapper

HERE = pathlib.Path().resolve()

## Step 0: Point to your file

Set `data_file` to the path of your measurement file.  The cell below
resolves the bundled tensile test example so the notebook runs out of the
box. Replace the path to use your own file.

In [3]:
# Replace this with your own file path, e.g.:
#   data_file = pathlib.Path('/path/to/my_test.TXT')

here = pathlib.Path().resolve()
data_file = here.parent / 'tests' / 'data' / 'example_tensile_test.TXT'
if not data_file.exists():
    data_file = here / 'tests' / 'data' / 'example_tensile_test.TXT'

print(f'Data file : {data_file.name}')
print(f'Exists    : {data_file.exists()}')

Data file : example_tensile_test.TXT
Exists    : True


## Step 1: Inspect the file

The example file is a Zwick/Roell testXpert III export.  It has two sections:

| Rows | Content |
|---|---|
| 1–20 | Header block: each row contains a label, a value, and optionally a unit |
| 21 | Column names row |
| 22 | Column units row (added by the instrument software; we skip it when reading) |
| 23+ | Numeric time-series data |

Run the cell below to see the exact structure before writing your mapping.

In [4]:
with open(data_file, encoding='utf-8') as fh:
    rows = list(csv.reader(fh, delimiter='\t', quotechar='"'))

print(f'Total rows: {len(rows)}\n')

print('=== Metadata block (rows 1\u201320) ===')
for i, row in enumerate(rows[:20]):
    label = row[0] if row else ''
    value = row[1] if len(row) > 1 else ''
    unit  = row[2] if len(row) > 2 else ''
    print(f'  {i+1:2d}: {label:<32}  {value:<15}  {unit}')

print(f'\n=== Column headers (row 21) ===')
print('  ', rows[20])

print(f'\n=== Column units (row 22) ===')
print('  ', rows[21])

print(f'\n=== First 3 data rows (rows 23\u201325) ===')
for i, row in enumerate(rows[22:25]):
    print(f'  {23+i}: {row}')

Total rows: 104

=== Metadata block (rows 1–20) ===
   1: Prüfinstitut                      institute_1      
   2: Projektnummer                     123456           
   3: Projektname                       DX56 Characterization  
   4: Datum/Uhrzeit                     44335.4          
   5: Maschinendaten                    Zwick Z100       
   6: Kraftaufnehmer                    KA-001           
   7: Wegaufnehmer                      WA-001           
   8: Prüfnorm                          ISO 6892-1       
   9: Werkstoff                         DX56             
  10: Probentyp                         flat             
  11: Prüfer                            J. Schmidt       
  12: Probenkennung 2                   DX56-A           
  13: Messlänge Standardweg             80               mm
  14: Versuchslänge                     120              mm
  15: Probendicke                       1.55             mm
  16: Probenbreite                      20.04            mm
  17: 

## Step 2: Write your mapping

The mapping has three sections:

**`file`**: controls how QuickMapper reads your file.
- `skip_rows`: number of rows to skip before the column names row
- `skip_after_header`: number of rows to skip right after the column names row (use this when
  the instrument inserts an extra row of unit labels below the column names, as in this file)

**`metadata`**: which fields to pull from the header block, and how to record them in the graph.
- `property`: the ontology property IRI to use as the RDF predicate
- `unit`: hardcode a unit — either a full QUDT IRI or a plain string (`'°C'`, `'mm'`) which is
  looked up in a built-in alias table and resolved to the corresponding QUDT IRI
- `unit_from_file: true`: read the unit string from column 3 of the metadata row; the string is
  resolved to a QUDT IRI using the same alias table

**`columns`**: which columns to annotate with ontology class IRIs and units.
- `iri`: ontology class IRI for this column's measurement type (optional)
- `unit`: hardcode a unit — full QUDT IRI or plain string, auto-resolved as above
- `unit_from_file: true`: read the unit string from the file's units row (the row that
  `skip_after_header` skips); resolved to a QUDT IRI automatically


In [5]:
mapping = {
    'label': 'DX56 tensile test — example',

    # ── File reading options ──────────────────────────────────────────
    'file': {
        'separator':         '\t',
        'encoding':          'utf-8',
        'skip_rows':          20,    # skip the 20 metadata rows before the header
        'skip_after_header':   1,    # skip the units row right after the header
    },

    # ── Metadata extraction ─────────────────────────────────────────────
    'metadata': {
        'rows': 20,
        'fields': {
            'Prüfnorm': {                            # test standard (e.g. ISO 6892-1)
                'property': 'http://purl.org/dc/terms/conformsTo',
            },
            'Temperatur': {                          # test temperature
                'property': 'https://w3id.org/pmd/co/PMD_0000071',
                'unit':     '°C',                    # plain string → resolved to QUDT DEG_C
            },
            'Prüfgeschwindigkeit': {                 # strain rate
                'property':      'https://w3id.org/pmd/tto/TTO_0000051',
                'unit_from_file': True,              # unit read from column 3 of this row ('mm/s')
            },
            'Messlänge Standardweg': {               # gauge length
                'property': 'https://w3id.org/pmd/tto/TTO_0000002',
                'unit':     'mm',                    # plain string → resolved to QUDT MilliM
            },
        },
    },

    # ── Column annotations ────────────────────────────────────────────────
    'columns': {
        'Prüfzeit': {                                # test time — no TTO v3 class IRI
            'unit': 's',                             # plain string → resolved to QUDT SEC
        },
        'Standardkraft': {                           # force — no TTO v3 class IRI
            'unit': 'N',                             # plain string → resolved to QUDT N
        },
        'Standardweg': {                             # extension
            'iri':  'https://w3id.org/pmd/tto/TTO_0000005',
            'unit': 'mm',                            # plain string → resolved to QUDT MilliM
        },
        'Dehnung': {                                 # elongation
            'iri':           'https://w3id.org/pmd/tto/TTO_0000004',
            'unit_from_file': True,                  # unit read from the file's units row ('%')
        },
    },
}

# Optionally save for reuse
mapping_file = HERE / 'my_mapping.yaml'
mapping_file.write_text(yaml.dump(mapping, allow_unicode=True), encoding='utf-8')
print(f'Mapping saved to {mapping_file.name}')

Mapping saved to my_mapping.yaml


## Step 3: Run the mapper

`QuickMapper` reads the file, extracts the metadata, applies the column
annotations, and builds an RDF graph.  No schema files or JSONata transforms
are needed.

In [6]:
mapper = QuickMapper(mapping)
result = mapper.run(data_file)

print('Mapping summary:')
print(json.dumps(result.oold_doc, indent=2))

QuickMapper: unit resolution for 'example_tensile_test.TXT':
  resolved   '%'              ->  http://qudt.org/vocab/unit/PERCENT
  resolved   'mm/s'           ->  http://qudt.org/vocab/unit/MilliM-PER-SEC
Mapping summary:
{
  "id": "https://example.org/datasets/example_tensile_test",
  "type": "http://www.w3.org/ns/csvw#Table",
  "label": "DX56 tensile test \u2014 example",
  "source": "example_tensile_test.TXT",
  "metadata": {
    "Pr\u00fcfnorm": {
      "value": "ISO 6892-1"
    },
    "Temperatur": {
      "value": "22",
      "unit": "http://qudt.org/vocab/unit/DEG_C"
    },
    "Pr\u00fcfgeschwindigkeit": {
      "value": "0.1",
      "unit": "http://qudt.org/vocab/unit/MilliM-PER-SEC"
    },
    "Messl\u00e4nge Standardweg": {
      "value": "80",
      "unit": "http://qudt.org/vocab/unit/MilliM"
    }
  },
  "columns": {
    "Pr\u00fcfzeit": {
      "unit": "http://qudt.org/vocab/unit/SEC"
    },
    "Dehnung": {
      "iri": "https://w3id.org/pmd/tto/TTO_0000004",
      "uni

### Unit resolution

QuickMapper automatically resolves unit strings to QUDT IRIs using a built-in
lookup table.  Resolution is applied in three places:

- **Metadata** `unit_from_file: true`: the unit string is read from column 3
  of the metadata row and looked up.
- **Metadata** plain-string `unit:` (e.g. `'°C'`, `'mm'`): resolved before
  the triple is written.
- **Column** `unit_from_file: true`: the unit string is read from the file's
  units row (the row that `skip_after_header` skips) and looked up.
- **Column** plain-string `unit:` (e.g. `'s'`, `'N'`, `'mm'`): same
  resolution applies.

All resolutions are recorded in `result.oold_doc["unit_resolutions"]`.  A
value of `null` means the string was not recognised and was stored as a plain
string instead of a linked IRI.


In [7]:
result.oold_doc["unit_resolutions"]

{'%': 'http://qudt.org/vocab/unit/PERCENT',
 'mm/s': 'http://qudt.org/vocab/unit/MilliM-PER-SEC'}

The example below uses a domain-specific unit string that is not in the
built-in table (`"HBW 10/3000"` is a Brinell hardness notation) to show
exactly what an unrecognised unit looks like in the output.

In [8]:
import tempfile

sample_content = (
    '"Hardness"\t245.0\t"HBW 10/3000"\n'  # metadata: label | value | unit
    '"Value"\n'                              # column names row
    '1.0\n'                                  # data row
)
with tempfile.NamedTemporaryFile(mode='w', suffix='.tsv', delete=False, encoding='utf-8') as f:
    f.write(sample_content)
    sample_file = pathlib.Path(f.name)

unrecognised_result = QuickMapper({
    "columns": {},
    "file": {"skip_rows": 1, "separator": "\t"},
    "metadata": {
        "rows": 1,
        "fields": {
            "Hardness": {"property": "http://example.org/hardness", "unit_from_file": True},
        },
    },
}).run(sample_file)

sample_file.unlink()
print(unrecognised_result.oold_doc["unit_resolutions"])

QuickMapper: unit resolution for 'tmpfiba7jt2.tsv':
  not found  'HBW 10/3000'    ->  stored as plain string
{'HBW 10/3000': None}


## Step 4: Inspect extracted metadata

Each configured metadata field has been extracted from the file's header block
and is available in `result.oold_doc["metadata"]`.  Fields with a unit are
stored as a value/unit pair in the graph.

In [9]:
meta = result.oold_doc['metadata']
print(f'Extracted {len(meta)} metadata fields:\n')
for label, info in meta.items():
    unit = info.get('unit', '')
    print(f'  {label:<30}  value={info["value"]!r:<12}  unit={unit!r}')

Extracted 4 metadata fields:

  Prüfnorm                        value='ISO 6892-1'  unit=''
  Temperatur                      value='22'          unit='http://qudt.org/vocab/unit/DEG_C'
  Prüfgeschwindigkeit             value='0.1'         unit='http://qudt.org/vocab/unit/MilliM-PER-SEC'
  Messlänge Standardweg           value='80'          unit='http://qudt.org/vocab/unit/MilliM'


## Step 5: Inspect the time-series data

The full DataFrame is in `result.dataframe`.  All columns are present,
including any that were not annotated.

In [10]:
df = result.dataframe
print(f'{len(df)} rows \u00d7 {len(df.columns)} columns\n')
df.head()

82 rows × 6 columns



,Prüfzeit,Standardkraft,Traversenweg absolut,Standardweg,Breitenänderung,Dehnung
0,0.0,0.0,0.000,0.00,-0.000,0.00
1,0.1,807.6,0.011,0.01,-0.001,0.01
2,0.2,1615.2,0.023,0.02,-0.001,0.02
3,0.3,2422.8,0.034,0.03,-0.002,0.04
4,0.5,3230.4,0.045,0.04,-0.003,0.05


In [11]:
print(f'{"Column":<30}  {"Class (short)":<35}  Unit IRI')
print('-' * 95)
for col in df.columns:
    iri  = result.column_iris.get(col)
    unit = result.column_units.get(col, '')
    cls  = iri.rsplit('/', 1)[-1] if iri and '/' in iri else (iri or '—')
    print(f'{col:<30}  {cls:<35}  {unit}')


Column                          Class (short)                        Unit IRI
-----------------------------------------------------------------------------------------------
Prüfzeit                        —                                    http://qudt.org/vocab/unit/SEC
Standardkraft                   —                                    http://qudt.org/vocab/unit/N
Traversenweg absolut            —                                    
Standardweg                     TTO_0000005                          http://qudt.org/vocab/unit/MilliM
Breitenänderung                 —                                    
Dehnung                         TTO_0000004                          http://qudt.org/vocab/unit/PERCENT


## Step 6: Inspect the RDF graph

The graph contains:
- One root node typed as `csvw:Table`, with title and source labels
- One blank node per extracted metadata field (value + unit triple when a unit is present)
- One `csvw:Column` descriptor node per annotated column, with an optional
  ontology class type and a unit IRI via `obo:IAO_0000039`


In [12]:
flat = result.flat_graph
print(f'Graph: {len(flat)} triples\n')
print(flat.serialize(format='turtle'))

Graph: 36 triples

@prefix csvw: <http://www.w3.org/ns/csvw#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix ns1: <http://purl.obolibrary.org/obo/> .
@prefix ns2: <https://w3id.org/pmd/co/> .
@prefix ns3: <https://w3id.org/pmd/tto/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/datasets/example_tensile_test> a csvw:Table ;
    rdfs:label "DX56 tensile test — example" ;
    dcterms:conformsTo "ISO 6892-1" ;
    dcterms:source "example_tensile_test.TXT" ;
    dcterms:title "DX56 tensile test — example" ;
    csvw:column <https://example.org/datasets/example_tensile_test/Dehnung>,
        <https://example.org/datasets/example_tensile_test/Prüfzeit>,
        <https://example.org/datasets/example_tensile_test/Standardkraft>,
        <https://example.org/datasets/example_tensile_test/Standardweg> ;
    ns2:PMD_0000071 [ ns1:IAO_0000039 <h

## Step 7: Save

Both outputs are written next to this notebook.

In [13]:
stem = data_file.stem

ttl_path = HERE / f'{stem}.ttl'
flat.serialize(destination=str(ttl_path), format='turtle')
print(f'RDF written to       {ttl_path.name}')

try:
    parquet_path = HERE / f'{stem}.parquet'
    df.to_parquet(parquet_path, index=False)
    print(f'DataFrame written to {parquet_path.name}')
except ImportError:
    csv_path = HERE / f'{stem}_data.csv'
    df.to_csv(csv_path, index=False)
    print(f'DataFrame written to {csv_path.name}  (install pyarrow for Parquet)')

RDF written to       example_tensile_test.ttl
DataFrame written to example_tensile_test.parquet


## Summary

| Step | What happens |
|---|---|
| 0 | Point to your file |
| 1 | Inspect the file structure (metadata block + column headers + data) |
| 2 | Write a mapping config (`metadata` fields + `columns` annotations) |
| 3 | Run `QuickMapper` to produce the RDF graph |
| 4 | Inspect extracted metadata |
| 5 | Explore the DataFrame |
| 6 | Inspect the RDF graph |
| 7 | Save graph as `.ttl` and data as `.parquet` |

## When to graduate to a formal schema

The `QuickMapper` output is a good starting point for exploration and
one-off conversions.  When the same file type recurs regularly, consider
formalising it:

1. **Add a parser** in `parsers/` following the
   [adding-a-parser guide](2_adding-a-parser.md).  The `column_mapping.json`
   is essentially the `columns:` block from your mapping config.
2. **Add a schema** in `semantic-schemas` for the domain, with a full
   OO-LD context and SHACL shapes for validation.
3. **Replace `QuickMapper`** with a `Transformer` + your new parser to
   get ontology-aligned, validated RDF.